In [9]:
from pathlib import Path
import mne
import pandas as pd


data = Path(r"C:\Users\carlo\OneDrive - Universidade de Lisboa\Documents\GitHub\heart-lung-brain-coupling-for-RBD\Data\cap-sleep-database-1.0.0\n1.edf")
txt_path = r"C:\Users\carlo\OneDrive - Universidade de Lisboa\Documents\GitHub\heart-lung-brain-coupling-for-RBD\Data\cap-sleep-database-1.0.0\n1.txt"

raw = mne.io.read_raw_edf(data, preload=True, verbose=False)

print(raw.info["sfreq"])
print(raw.ch_names[:10])

C:\Users\carlo\AppData\Local\Temp\ipykernel_32204\3252767969.py:9: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(data, preload=True, verbose=False)
C:\Users\carlo\AppData\Local\Temp\ipykernel_32204\3252767969.py:9: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(data, preload=True, verbose=False)
C:\Users\carlo\AppData\Local\Temp\ipykernel_32204\3252767969.py:9: RuntimeWarning: Highpass cutoff frequency 10.0 is greater than lowpass cutoff frequency 3.0, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(data, preload=True, verbose=False)


512.0
['ROC-LOC', 'LOC-ROC', 'F2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'F1-F3', 'F3-C3', 'C3-P3', 'P3-O1']


In [4]:
!pip install h5py


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached h5py-3.16.0-cp312-cp312-win_amd64.whl.metadata (3.1 kB)
Using cached h5py-3.16.0-cp312-cp312-win_amd64.whl (3.2 MB)


In [27]:
import importlib
import functions
import script

importlib.reload(script)

from script import process_patient_to_hdf5
from functions import *

In [ ]:
df = pd.read_csv(
    txt_path,
    sep='\t',
    header=None,
    names=["Sleep Stage", "Position", "Time [hh:mm:ss]", "Event", "Duration[s]", "Location"],
    skiprows=22
)

df["Duration[s]"] = pd.to_numeric(df["Duration[s]"], errors="coerce")
df = df[df["Event"].astype(str).str.startswith("SLEEP-", na=False)].copy()
df = df[df["Duration[s]"] == 30].copy()
df = df[df["Sleep Stage"].astype(str).isin(["W", "R", "S1", "S2", "S3", "S4"])].reset_index(drop=True)

print("Rows after filtering:", len(df))
print(df["Sleep Stage"].value_counts())

df.head()

,Sleep Stage,Position,Time [hh:mm:ss],Event,Duration[s],Location
0,W,Unknown Position,22:09:33,SLEEP-S0,30,ROC-LOC
1,W,Unknown Position,22:10:03,SLEEP-S0,30,ROC-LOC
2,W,Unknown Position,22:10:33,SLEEP-S0,30,ROC-LOC
3,W,Unknown Position,22:11:03,SLEEP-S0,30,ROC-LOC
4,W,Unknown Position,22:11:33,SLEEP-S0,30,ROC-LOC


In [28]:
process_patient_to_hdf5(
    raw=raw,
    patient_id="n01",
    group="control",
    df=df,
    output_dir=r"C:\Users\carlo\OneDrive - Universidade de Lisboa\Documents\GitHub\heart-lung-brain-coupling-for-RBD\HDF files"
)

BP DF length: 1152
Saved → C:\Users\carlo\OneDrive - Universidade de Lisboa\Documents\GitHub\heart-lung-brain-coupling-for-RBD\HDF files\n01.h5


In [29]:
import h5py

h5_path = r"C:\Users\carlo\OneDrive - Universidade de Lisboa\Documents\GitHub\heart-lung-brain-coupling-for-RBD\HDF files\n01.h5"

with h5py.File(h5_path, "r") as f:
    print(list(f.keys()))                 # should show ['n01']
    print(list(f["n01"]["epochs"].keys())[:5])  # first epochs

['n01']
['epoch_0000', 'epoch_0001', 'epoch_0002', 'epoch_0003', 'epoch_0004']


In [ ]:
from pathlib import Path
import h5py
import numpy as np
import pandas as pd

patient_id = "n01"
candidate_paths = [
    Path("../HDF files") / f"{patient_id}.h5",
    Path("HDF files") / f"{patient_id}.h5",
    Path(r"C:\Users\carlo\OneDrive - Universidade de Lisboa\Documents\GitHub\heart-lung-brain-coupling-for-RBD\HDF files\n01.h5"),
]

path = next((p for p in candidate_paths if p.exists()), candidate_paths[-1])
print("Exists:", path.exists())
print("Path:", path.resolve())


In [ ]:
with h5py.File(path, "r") as f:
    print("Top level keys:", list(f.keys()))

    patient = f[patient_id]
    print("Attrs:", dict(patient.attrs))
    print("Subgroups:", list(patient.keys()))

    epoch_keys = list(patient["epochs"].keys())
    print("N epochs:", len(epoch_keys))
    print("First 5 epochs:", epoch_keys[:5])

    ep = patient["epochs"][epoch_keys[0]]
    stage = ep["sleep_stage"][()]
    if isinstance(stage, bytes):
        stage = stage.decode()

    print("\nFirst epoch summary")
    print("start_time:", ep["start_time"][()])
    print("end_time:", ep["end_time"][()])
    print("sleep_stage:", stage)
    print("delta:", ep["eeg_bandpower"]["delta"][()])
    print("hr_mean_bpm:", ep["hrv"]["hr_mean_bpm"][()])
    print("cpc ratio:", ep["cpc"]["ratio"][()])
    print("pearson_1s_r:", ep["hep"]["pearson_1s_r"][()])
    print("hep_30s:", ep["hep"]["hep_30s"][()])
    print("waveform mean shape:", ep["waveform"]["mean"]["signal"].shape)


In [ ]:
rows = []

with h5py.File(path, "r") as f:
    epochs = f[patient_id]["epochs"]

    for ep_name in epochs.keys():
        ep = epochs[ep_name]
        stage = ep["sleep_stage"][()]
        if isinstance(stage, bytes):
            stage = stage.decode()

        rows.append({
            "epoch": ep_name,
            "start": ep["start_time"][()],
            "end": ep["end_time"][()],
            "stage": stage,
            "delta": ep["eeg_bandpower"]["delta"][()],
            "theta": ep["eeg_bandpower"]["theta"][()],
            "alpha": ep["eeg_bandpower"]["alpha"][()],
            "beta": ep["eeg_bandpower"]["beta"][()],
            "gamma": ep["eeg_bandpower"]["gamma"][()],
            "n_beats": ep["hrv"]["n_beats"][()],
            "hr_mean_bpm": ep["hrv"]["hr_mean_bpm"][()],
            "rmssd_ms": ep["hrv"]["rmssd_ms"][()],
            "sdnn_ms": ep["hrv"]["sdnn_ms"][()],
            "pnn50_pct": ep["hrv"]["pnn50_pct"][()],
            "hfc": ep["cpc"]["hfc"][()],
            "lfc": ep["cpc"]["lfc"][()],
            "cpc_ratio": ep["cpc"]["ratio"][()],
            "pearson_1s_r": ep["hep"]["pearson_1s_r"][()],
            "pearson_1s_p": ep["hep"]["pearson_1s_p"][()],
            "spearman_1s_r": ep["hep"]["spearman_1s_r"][()],
            "spearman_1s_p": ep["hep"]["spearman_1s_p"][()],
            "hep_30s": ep["hep"]["hep_30s"][()],
            "delta_30s": ep["hep"]["delta_30s"][()],
        })

df_check = pd.DataFrame(rows)
display(df_check.head())
display(df_check.describe(include="all"))
print("\nNaN count per column:")
print(df_check.isna().sum())

print("\nEpoch durations:", df_check["end"].sub(df_check["start"]).unique())
print("Stages:", sorted(df_check["stage"].dropna().unique()))
print("Rows with HEP 1s missing:", df_check["pearson_1s_r"].isna().sum())
print("Rows with CPC missing:", df_check["cpc_ratio"].isna().sum())

df_check[["epoch", "stage", "delta", "hr_mean_bpm", "cpc_ratio", "pearson_1s_r", "hep_30s", "delta_30s"]].head(10)
